In [1]:
import ExTRA as ex
import numpy as np
import astroquery
import astropy
import matplotlib.pyplot as plt

ExTRA  imported correctly


## This notebook shows how to get a consistent single stellar model between gaia and hipparcos data.
## we propagate the single stellar model in cartesian coordinates when given a radial velocity in km/s as accurate as possible. This way the secular acceleration BETWEEN THE MODELS is eliminated


## Step 1:
## Correcting HIP to be consistent with Gaia given a radial velocity
### as for this example we use REAL HIP and GDR3 data of Nu Octantis

In [30]:
#Nu Oct as example
#HIP107089
hip_sss=np.array([5.6787539579,-1.3507009230,47.16,66.40,-239.10])
hip_iad,t_hip=ex.hip_read("data/nu_oct/HIP107089_esa.d")



In [31]:
gaia_asc=(21 +41/60 +28.5420355132/3600 )*15 #deg
gaia_dec=(-77 -23/60 -24.031541832/3600) #deg


gaia_sss=np.array([np.radians(gaia_asc),np.radians(gaia_dec),51.5172,68.656,-250.044]) # in radians for propagator

v_rad_gaia=34.4 #km/s


In [4]:
propagator=ex.EpochPropagation() #this class is able to propagate astrometry given v_rad and 2 standard epochs in julian YEARS

In [5]:
#Notice i give gaia model absolute position in RADIANS
#input of the propagator is [rad],[rad],[mas],[mas/yr],[mas/yr],[km/s],[yr],[yr]
gaia1991=np.array(propagator.propagate_astrometry(*gaia_sss,v_rad_gaia,1991.25,2016.0))
gaia1991_sss=gaia1991[:5]
print(gaia1991_sss)


[   5.67879677   -1.35074046   51.51488912   68.65904869 -250.01903991]


In [39]:
gaia_sss

array([   5.67875903,   -1.35071046,   51.5172    ,   68.656     ,
       -250.044     ])

### Now that we have gaia1991, we can correct our hipparcos residuals to it, so models between HIP astrometric data and gaia astrometric data are correct.

In [7]:
gaia1991_sss_deg=gaia1991_sss.copy()
hip_sss_deg=hip_sss.copy()

#abs pos in degrees
gaia1991_sss_deg[:2]=np.degrees(gaia1991_sss[:2])
hip_sss_deg[:2]=np.degrees(hip_sss[:2])

#input of my abs res func is [deg],[deg],[mas],[mas/yr],[mas/yr]
abs_consistent=ex.abs_res(hip_iad[-2],gaia1991_sss_deg,hip_sss_deg,hip_iad)

In [38]:
#cartesian velocities in km/s:
mu_1991=np.array([gaia1991[3],gaia1991[4],gaia1991[5]])
v_1991=ex.mu_to_v(gaia1991[2],mu_1991)

v_rad_1991=v_1991[-1]

triad_1991=np.array(ex.normal_triad(gaia1991[0],gaia1991[1]))
v_cartesian=triad_1991.T @ v_1991
delta_t=v_cartesian*(365.25*24*60**2)/ex.lightyear
ex.cartesian_to_spherical(*delta_t)[0]*365.25*60*24



np.float64(73.4480604560982)

## Step 2: lighttime correction on timestamps

In [21]:
#i propagate astrometry between jyears, so i need to convert the hipparcos measurement times to julian year, same will have to be done to gaia
from astropy.time import Time
t_hip_jyear_relative = Time(t_hip,format="jd").jyear -1991.25 #timestamps relative to sepoch in julianYEARS !!!!!
#ltd correction:

tau_ltd=[]
for t in t_hip_jyear_relative:
    
    tau_ltd.append(ex.ltd_accurate(t,gaia1991_sss,v_rad_1991))
tau_ltd=np.array(tau_ltd)/(ex.julian_year_seconds)


t_hip_corrected=t_hip_jyear_relative+tau_ltd


#if we dont use cartesian velocity and only the radial velocity:
tau_approx=ex.ltd_approx(v_rad_1991)*t_hip_jyear_relative 

#actually the HIP measurement times are given in jyears relative to j1991.25 in the original file.
#its easier to correct HIP than gaia since gaia will give JD timestamps in its original file
#Since i already compute the timestamps in julian days from the hip data,
#gaia as well as hip measurement times will have to be computed into julian year format relative to their respective standard epochs.
#this sounds complicated, but at the end of the day it will just be a one liner like above using astropy.time function.

## Step 3) correcting for secular acceleration

In [22]:
#correceting the hipparcos data is as easy as adding the amounted change onto the standard model of 1991 
#and then removing the shift according to the scan angle
shift_asc,shift_dec=ex.secular_shift(gaia1991[:-1],v_rad_1991,t_hip_corrected)
#please see that we used the already ltd corrected timestamps

#removing the shift accordingly:
abs_secular_corrected=abs_consistent-(shift_asc*hip_iad[0]+shift_dec*hip_iad[1])

In [23]:
secular_change=ex.secular_acceleration(gaia1991,v_rad_1991)
print("changes of par,mu_a and mu_d per year:",secular_change)
print("this is just a fun fact and not important for computing")

changes of par,mu_a and mu_d per year: [-9.33655751e-05 -9.06268919e-04  2.48875293e-04]
this is just a fun fact and not important for computing


## Here i compare to a Lindegren paper to see if our estimates are comparable, and they are:
#### (im using more accurate proper motions than he did)

In [24]:
#L. Lindegren 2021:
#The largest changes are expected for Barnards’s star
#(HIP 87937) owing to its sizeable parallax ('547 mas), proper
#motion ('10 393 mas yr−1), and radial velocity ('−110 km s−1).
#For this star, the perspective effects produce, over the 24.75 yr,
#a position difference of about 393 mas, an increase in the paral-
#lax by 0.84 mas, and an increase in the proper motion by about
#32 mas yr−1.

barnard=np.array([0,0,547,-801.551,10362.394])
v_barnard=-110
sec_lindegren=np.array(ex.secular_acceleration(barnard,v_barnard))* 24.75
shift_lindegren=ex.secular_shift(barnard,v_barnard,24.75)
print("change in parallax and proper motion over 24.75 years:",sec_lindegren)
print("total shift over 24.75 years:",shift_lindegren)

change in parallax and proper motion over 24.75 years: [ 0.83309776 31.56448697 -2.44157345]
total shift over 24.75 years: [390.61052623 -30.21447147]


## Overview

In [27]:
hip_iad_corrected=hip_iad.copy()
hip_iad_corrected[-2]=abs_secular_corrected


In [28]:
print("old likelihood:",ex.loglikelihood(hip_iad[-2],hip_iad[-1],0))
print("new likelihood:",ex.loglikelihood(hip_iad_corrected[-2],hip_iad[-1],0))

old likelihood: 530.8247862687826
new likelihood: 555.4973453199237


### the likelihood got worse as expected, since hipparcos on its own used to have the optimal fit with the data available. the now new residuals are consistent with gaias single steller solution and can therefor easily be fit in combination. the single stellar solution fit corrections can now be applied to both of the residuals in the same way

# FULL CORRECTOR FUNCTION: